In [3]:
!rm -rf ~/.cache/huggingface/hub/models--utter-project--EuroLLM-1.7B

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import unicodedata
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from pathlib import Path
import numpy as np

Mounted at /content/drive


In [5]:
from huggingface_hub import login
login()

In [6]:
#Some helper functions
from transformers import AutoModelForCausalLM

def load_causal_model(model_name):
    """Load an autoregressive/causal language model"""
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,  # Use half precision to save memory
        device_map='auto'  # use GPU
    )
    model.eval()
    print("Model loaded!")
    return tokenizer, model

def remove_accents(text): #remove accents from greek text
    nfd = unicodedata.normalize('NFD', text)
    return ''.join(char for char in nfd if unicodedata.category(char) != 'Mn')

In [7]:
class CausalLMScorer:

    def __init__(self, tokenizer, model):
        """Initialize the scorer"""
        self.tokenizer = tokenizer
        self.model = model

    def score_sentence(self, sentence):
        """Score a sentence using log-probability"""
        sentence = remove_accents(sentence.lower())
        input_ids = self.tokenizer.encode(sentence, return_tensors='pt')
        input_ids = input_ids.to(self.model.device)

        with torch.no_grad():
            outputs = self.model(input_ids, labels=input_ids)
            # outputs.loss is already the avg negative log-likelihood per token!
            avg_log_prob = -outputs.loss.item()

        num_tokens = input_ids.shape[1]
        avg_surprisal = -avg_log_prob

        return {
            'avg_log_prob': avg_log_prob,  #length-normalized
            'avg_surprisal': avg_surprisal,
            'num_tokens': num_tokens
        }

In [8]:
# Load model
model_name = "utter-project/EuroLLM-1.7B"
model_tokenizer, model = load_causal_model(model_name)

# Create scorer
model_scorer = CausalLMScorer(model_tokenizer, model)

# Test with a minimal pair
gram = "Το παιδί τρώει"
ungram = "Το παιδιά τρώει"

result_gram = model_scorer.score_sentence(gram)
result_ungram = model_scorer.score_sentence(ungram)

print(f"Grammatical: '{gram}'")
print(f"  PLL: {result_gram['avg_log_prob']:.2f}")
print(f"  Surprisal: {result_gram['avg_surprisal']:.2f}")

print(f"\nUngrammatical: '{ungram}'")
print(f"  PLL: {result_ungram['avg_log_prob']:.2f}")
print(f"  Surprisal: {result_ungram['avg_surprisal']:.2f}")

print(f"\nModel prefers grammatical: {result_gram['avg_log_prob'] > result_ungram['avg_log_prob']}")

Loading utter-project/EuroLLM-1.7B...


config.json:   0%|          | 0.00/637 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/960 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 2.41MB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.31GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded!
Grammatical: 'Το παιδί τρώει'
  PLL: -5.92
  Surprisal: 5.92

Ungrammatical: 'Το παιδιά τρώει'
  PLL: -6.07
  Surprisal: 6.07

Model prefers grammatical: True


In [9]:
def load_phenomenon_pairs(phenomenon_path):
    """Load pairs for one phenomenon"""
    gram_file = phenomenon_path / "correct.txt"
    ungram_file = phenomenon_path / "incorrect.txt"

    with open(gram_file, 'r', encoding='utf-8') as f:
        gram_sentences = f.readlines()

    with open(ungram_file, 'r', encoding='utf-8') as f:
        ungram_sentences = f.readlines()

    pairs = list(zip(gram_sentences, ungram_sentences))
    return pairs

def load_all_phenomena(data_dir):
    """Load all phenomena"""
    all_pairs = {}
    data_path = Path(data_dir)

    for phenomenon_folder in data_path.iterdir():
        if phenomenon_folder.is_dir():
            phenomenon_name = phenomenon_folder.name

            try:
                pairs = load_phenomenon_pairs(phenomenon_folder)
                all_pairs[phenomenon_name] = pairs
                print(f"✓ Loaded {phenomenon_name}: {len(pairs)} pairs")
            except FileNotFoundError:
                print(f"✗ Skipped {phenomenon_name}: files not found")

    return all_pairs

In [10]:
# test loading all phenomena
all_data = load_all_phenomena("/content/drive/MyDrive/Thesis/data/phenomena")

print(f"\nTotal phenomena loaded: {len(all_data)}")
print("\nSummary:")
for phen_name, pairs in all_data.items():
    print(f"  - {phen_name}: {len(pairs)} pairs")

✓ Loaded noun_adjective_agreement: 100 pairs
✓ Loaded aspect: 100 pairs
✓ Loaded einai_agreement: 100 pairs
✓ Loaded subject_verb_agreement: 100 pairs
✓ Loaded negations: 100 pairs
✓ Loaded case_selection: 100 pairs

Total phenomena loaded: 6

Summary:
  - noun_adjective_agreement: 100 pairs
  - aspect: 100 pairs
  - einai_agreement: 100 pairs
  - subject_verb_agreement: 100 pairs
  - negations: 100 pairs
  - case_selection: 100 pairs


In [11]:
# Evaluate with ALL metrics tracked
all_results = {}

for phenomenon_name, pairs in all_data.items():
    print(f"\n{'='*50}")
    print(f"Evaluating: {phenomenon_name}")
    print(f"{'='*50}")

    correct_count = 0
    total_count = len(pairs)
    pair_results = []

    for gram, ungram in pairs:
        # Score both
        gram_result = model_scorer.score_sentence(gram)
        ungram_result = model_scorer.score_sentence(ungram)

        # Compare using length-normalized score
        is_correct = gram_result['avg_log_prob'] > ungram_result['avg_log_prob']

        if is_correct:
            correct_count += 1

        # Store ALL metrics for analysis
        pair_results.append({
            'grammatical': gram.strip(),
            'ungrammatical': ungram.strip(),
            'gram_avg_log_prob': gram_result['avg_log_prob'],
            'ungram_avg_log_prob': ungram_result['avg_log_prob'],
            'gram_surprisal': gram_result['avg_surprisal'],
            'ungram_surprisal': ungram_result['avg_surprisal'],
            'gram_num_tokens': gram_result['num_tokens'],
            'ungram_num_tokens': ungram_result['num_tokens'],
            'correct': is_correct
        })

    # Calculate ALL average metrics
    accuracy = correct_count / total_count
    avg_gram_log_prob = sum(p['gram_avg_log_prob'] for p in pair_results) / len(pair_results)
    avg_ungram_log_prob = sum(p['ungram_avg_log_prob'] for p in pair_results) / len(pair_results)
    avg_gram_surprisal = sum(p['gram_surprisal'] for p in pair_results) / len(pair_results)
    avg_ungram_surprisal = sum(p['ungram_surprisal'] for p in pair_results) / len(pair_results)

    # Store everything
    all_results[phenomenon_name] = {
        'correct': correct_count,
        'total': total_count,
        'accuracy': accuracy,
        'avg_gram_log_prob': avg_gram_log_prob,
        'avg_ungram_log_prob': avg_ungram_log_prob,
        'avg_gram_surprisal': avg_gram_surprisal,
        'avg_ungram_surprisal': avg_ungram_surprisal,
        'pairs': pair_results
    }

    # Print ALL metrics
    print(f"Accuracy: {correct_count}/{total_count} = {accuracy:.2%}")
    print(f"Avg grammatical log prob: {avg_gram_log_prob:.2f}")
    print(f"Avg ungrammatical log prob: {avg_ungram_log_prob:.2f}")
    print(f"Avg grammatical surprisal: {avg_gram_surprisal:.2f}")
    print(f"Avg ungrammatical surprisal: {avg_ungram_surprisal:.2f}")

# Full summary table
print(f"\n{'='*90}")
print(f"COMPLETE SUMMARY")
print(f"{'='*90}")
print(f"{'Phenomenon':<25} {'Accuracy':<12} {'Gram LogP':<12} {'Ungram LogP':<12} {'Gram Surp':<12}")
print(f"{'-'*90}")
for phen, result in all_results.items():
    print(f"{phen:<25} {result['accuracy']:>10.2%} "
          f"{result['avg_gram_log_prob']:>11.2f} "
          f"{result['avg_ungram_log_prob']:>11.2f} "
          f"{result['avg_gram_surprisal']:>11.2f}")

total_correct = sum(r['correct'] for r in all_results.values())
total_pairs = sum(r['total'] for r in all_results.values())
overall_acc = total_correct / total_pairs
print(f"{'-'*90}")
print(f"{'Overall':<25} {overall_acc:>10.2%} ({total_correct}/{total_pairs})")


Evaluating: noun_adjective_agreement
Accuracy: 83/100 = 83.00%
Avg grammatical log prob: -4.56
Avg ungrammatical log prob: -4.77
Avg grammatical surprisal: 4.56
Avg ungrammatical surprisal: 4.77

Evaluating: aspect
Accuracy: 68/100 = 68.00%
Avg grammatical log prob: -3.80
Avg ungrammatical log prob: -3.93
Avg grammatical surprisal: 3.80
Avg ungrammatical surprisal: 3.93

Evaluating: einai_agreement
Accuracy: 87/100 = 87.00%
Avg grammatical log prob: -4.43
Avg ungrammatical log prob: -4.72
Avg grammatical surprisal: 4.43
Avg ungrammatical surprisal: 4.72

Evaluating: subject_verb_agreement
Accuracy: 95/100 = 95.00%
Avg grammatical log prob: -3.70
Avg ungrammatical log prob: -3.92
Avg grammatical surprisal: 3.70
Avg ungrammatical surprisal: 3.92

Evaluating: negations
Accuracy: 98/100 = 98.00%
Avg grammatical log prob: -4.20
Avg ungrammatical log prob: -4.65
Avg grammatical surprisal: 4.20
Avg ungrammatical surprisal: 4.65

Evaluating: case_selection
Accuracy: 79/100 = 79.00%
Avg gramma

In [12]:
import random

def sample_size_stability(scorer, all_data, sample_sizes=[25, 50, 75, 100], n_repeats=5):
    """
    For each phenomenon, score every pair ONCE, then resample at different
    sizes to see how the accuracy estimate changes with more pairs.
    """
    results = {}

    for phenomenon, pairs in all_data.items():
        print(f"\nPhenomenon: {phenomenon} ({len(pairs)} pairs available)")

        # Score every pair once — reuse for all sample sizes
        pair_correctness = []
        for gram, ungram in pairs:
            g = scorer.score_sentence(gram)
            u = scorer.score_sentence(ungram)
            pair_correctness.append(g['avg_log_prob'] > u['avg_log_prob'])

        phen_results = {}
        for size in sample_sizes:
            if size > len(pair_correctness):
                print(f"  ⚠ Skipping n={size}: only {len(pair_correctness)} pairs available")
                continue

            accs = []
            for rep in range(n_repeats):
                random.seed(rep)
                sample = random.sample(pair_correctness, size)
                accs.append(sum(sample) / size)

            phen_results[size] = {
                'mean_accuracy': sum(accs) / len(accs),
                'min_accuracy': min(accs),
                'max_accuracy': max(accs),
                'all_runs': accs
            }
            print(f"  n={size}: mean={phen_results[size]['mean_accuracy']:.2%} "
                  f"(range: {phen_results[size]['min_accuracy']:.2%}-{phen_results[size]['max_accuracy']:.2%})")

        results[phenomenon] = phen_results

    return results

# Run it
print("="*60)
print("SAMPLE-SIZE STABILITY EXPERIMENT")
print("="*60)
stability_results = sample_size_stability(model_scorer, all_data)

SAMPLE-SIZE STABILITY EXPERIMENT

Phenomenon: noun_adjective_agreement (100 pairs available)
  n=25: mean=80.80% (range: 76.00%-92.00%)
  n=50: mean=82.40% (range: 76.00%-90.00%)
  n=75: mean=82.40% (range: 81.33%-84.00%)
  n=100: mean=83.00% (range: 83.00%-83.00%)

Phenomenon: aspect (100 pairs available)
  n=25: mean=68.00% (range: 60.00%-76.00%)
  n=50: mean=70.00% (range: 66.00%-74.00%)
  n=75: mean=69.87% (range: 68.00%-72.00%)
  n=100: mean=68.00% (range: 68.00%-68.00%)

Phenomenon: einai_agreement (100 pairs available)
  n=25: mean=85.60% (range: 80.00%-96.00%)
  n=50: mean=86.00% (range: 80.00%-92.00%)
  n=75: mean=87.73% (range: 85.33%-90.67%)
  n=100: mean=87.00% (range: 87.00%-87.00%)

Phenomenon: subject_verb_agreement (100 pairs available)
  n=25: mean=94.40% (range: 92.00%-96.00%)
  n=50: mean=94.40% (range: 92.00%-98.00%)
  n=75: mean=95.73% (range: 94.67%-97.33%)
  n=100: mean=95.00% (range: 95.00%-95.00%)

Phenomenon: negations (100 pairs available)
  n=25: mean=96.00%

In [13]:
# Evaluate with ALL metrics tracked
all_results = {}

for phenomenon_name, pairs in all_data.items():
    print(f"\n{'='*50}")
    print(f"Evaluating: {phenomenon_name}")
    print(f"{'='*50}")

    correct_count = 0
    total_count = len(pairs)
    pair_results = []

    for gram, ungram in pairs:
        gram_result = model_scorer.score_sentence(gram)
        ungram_result = model_scorer.score_sentence(ungram)

        is_correct = gram_result['avg_log_prob'] > ungram_result['avg_log_prob']
        if is_correct:
            correct_count += 1

        pair_results.append({
            'grammatical': gram.strip(),
            'ungrammatical': ungram.strip(),
            'gram_avg_log_prob': gram_result['avg_log_prob'],
            'ungram_avg_log_prob': ungram_result['avg_log_prob'],
            'gram_surprisal': gram_result['avg_surprisal'],
            'ungram_surprisal': ungram_result['avg_surprisal'],
            'gram_num_tokens': gram_result['num_tokens'],
            'ungram_num_tokens': ungram_result['num_tokens'],
            'correct': is_correct
        })

    accuracy = correct_count / total_count
    avg_gram_log_prob = sum(p['gram_avg_log_prob'] for p in pair_results) / len(pair_results)
    avg_ungram_log_prob = sum(p['ungram_avg_log_prob'] for p in pair_results) / len(pair_results)
    avg_gram_surprisal = sum(p['gram_surprisal'] for p in pair_results) / len(pair_results)
    avg_ungram_surprisal = sum(p['ungram_surprisal'] for p in pair_results) / len(pair_results)

    all_results[phenomenon_name] = {
        'correct': correct_count,
        'total': total_count,
        'accuracy': accuracy,
        'avg_gram_log_prob': avg_gram_log_prob,
        'avg_ungram_log_prob': avg_ungram_log_prob,
        'avg_gram_surprisal': avg_gram_surprisal,
        'avg_ungram_surprisal': avg_ungram_surprisal,
        'pairs': pair_results
    }

    print(f"Accuracy: {correct_count}/{total_count} = {accuracy:.2%}")
    print(f"Avg grammatical log prob: {avg_gram_log_prob:.2f}")
    print(f"Avg ungrammatical log prob: {avg_ungram_log_prob:.2f}")
    print(f"Avg grammatical surprisal: {avg_gram_surprisal:.2f}")
    print(f"Avg ungrammatical surprisal: {avg_ungram_surprisal:.2f}")

# Full summary table
print(f"\n{'='*90}")
print(f"COMPLETE SUMMARY")
print(f"{'='*90}")
print(f"{'Phenomenon':<25} {'Accuracy':<12} {'Gram LogP':<12} {'Ungram LogP':<12} {'Gram Surp':<12}")
print(f"{'-'*90}")
for phen, result in all_results.items():
    print(f"{phen:<25} {result['accuracy']:>10.2%} "
          f"{result['avg_gram_log_prob']:>11.2f} "
          f"{result['avg_ungram_log_prob']:>11.2f} "
          f"{result['avg_gram_surprisal']:>11.2f}")

total_correct = sum(r['correct'] for r in all_results.values())
total_pairs = sum(r['total'] for r in all_results.values())
overall_acc = total_correct / total_pairs
print(f"{'-'*90}")
print(f"{'Overall':<25} {overall_acc:>10.2%} ({total_correct}/{total_pairs})")


Evaluating: noun_adjective_agreement
Accuracy: 83/100 = 83.00%
Avg grammatical log prob: -4.56
Avg ungrammatical log prob: -4.77
Avg grammatical surprisal: 4.56
Avg ungrammatical surprisal: 4.77

Evaluating: aspect
Accuracy: 68/100 = 68.00%
Avg grammatical log prob: -3.80
Avg ungrammatical log prob: -3.93
Avg grammatical surprisal: 3.80
Avg ungrammatical surprisal: 3.93

Evaluating: einai_agreement
Accuracy: 87/100 = 87.00%
Avg grammatical log prob: -4.43
Avg ungrammatical log prob: -4.72
Avg grammatical surprisal: 4.43
Avg ungrammatical surprisal: 4.72

Evaluating: subject_verb_agreement
Accuracy: 95/100 = 95.00%
Avg grammatical log prob: -3.70
Avg ungrammatical log prob: -3.92
Avg grammatical surprisal: 3.70
Avg ungrammatical surprisal: 3.92

Evaluating: negations
Accuracy: 98/100 = 98.00%
Avg grammatical log prob: -4.20
Avg ungrammatical log prob: -4.65
Avg grammatical surprisal: 4.20
Avg ungrammatical surprisal: 4.65

Evaluating: case_selection
Accuracy: 79/100 = 79.00%
Avg gramma

In [14]:
import json
from pathlib import Path
from datetime import datetime

output_dir = Path("/content/drive/MyDrive/Thesis/results/autoregressive")
output_dir.mkdir(parents=True, exist_ok=True)

# --- Save full evaluation results ---
total_correct = sum(r['correct'] for r in all_results.values())
total_pairs = sum(r['total'] for r in all_results.values())
overall_acc = total_correct / total_pairs

results_to_save = {
    'model': model_name,
    'model_type': 'causal_lm',
    'timestamp': datetime.now().isoformat(),
    'overall': {
        'total_pairs': total_pairs,
        'total_correct': total_correct,
        'accuracy': overall_acc
    },
    'per_phenomenon': {
        phen: {
            'num_pairs': r['total'],
            'correct': r['correct'],
            'accuracy': r['accuracy'],
            'avg_grammatical_log_prob': r['avg_gram_log_prob'],
            'avg_ungrammatical_log_prob': r['avg_ungram_log_prob'],
            'avg_grammatical_surprisal': r['avg_gram_surprisal'],
            'avg_ungrammatical_surprisal': r['avg_ungram_surprisal'],
            'pairs': r['pairs']
        }
        for phen, r in all_results.items()
    }
}

results_file = output_dir / f"{model_name.replace('/', '_')}_results.json"
with open(results_file, 'w', encoding='utf-8') as f:
    json.dump(results_to_save, f, ensure_ascii=False, indent=2)
print(f"✓ Full results saved to {results_file}")

# --- Save stability results ---
stability_file = output_dir / f"{model_name.replace('/', '_')}_stability.json"
with open(stability_file, 'w', encoding='utf-8') as f:
    json.dump(stability_results, f, ensure_ascii=False, indent=2)
print(f"✓ Stability results saved to {stability_file}")

# --- Verify ---
print(f"\nresults file exists: {results_file.exists()}")
print(f"stability file exists: {stability_file.exists()}")

✓ Full results saved to /content/drive/MyDrive/Thesis/results/autoregressive/utter-project_EuroLLM-1.7B_results.json
✓ Stability results saved to /content/drive/MyDrive/Thesis/results/autoregressive/utter-project_EuroLLM-1.7B_stability.json

results file exists: True
stability file exists: True
